In [1]:
# Standard Library
import os
import re
import sys
import csv
import subprocess
import shutil
import random
import yaml
import piexif
from pathlib import Path
from collections import Counter

# Scientific & Visualization 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import cv2

# Deep Learning
import torch
from ultralytics import YOLO

# Dataset & Detection
from roboflow import Roboflow
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

In [2]:
# Project Root (adjust to your local setup)
PROJECT_ROOT = os.path.expanduser("~/Documents/Thesis")

# Core Directories
DATA_DIR   = os.path.join(PROJECT_ROOT, "data")
RUNS_DIR   = os.path.join(PROJECT_ROOT, "runs")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")

# Create Directories
for d in [DATA_DIR, RUNS_DIR, MODELS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f"Project root  : {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")
print(f"Runs directory: {RUNS_DIR}")

Project root  : C:\Users\Admin/Documents/Thesis
Data directory: C:\Users\Admin/Documents/Thesis\data
Runs directory: C:\Users\Admin/Documents/Thesis\runs


---

## SHOW EXIF METADATA

In [9]:
"""
show_exif_metadata.py

Displays ALL available EXIF metadata for a single image (e.g. a CoralNet
GoPro photo). Run this in a Jupyter code cell -- just set IMAGE_PATH below.

Uses two libraries for maximum coverage:
  - Pillow (PIL): standard EXIF tags, already installed most places
  - exifread:     picks up additional/maker-note tags PIL sometimes misses
                   (GoPro maker notes can carry extra lens/FOV/GPS info)

If exifread isn't installed, the script will still run using PIL alone --
it just prints a note and skips that section.
"""


def ensure_exifread():
    """Try to import exifread; install it quietly if missing."""
    try:
        import exifread  # noqa: F401
        return True
    except ImportError:
        try:
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "exifread", "--quiet",
                 "--break-system-packages"]
            )
            import exifread  # noqa: F401
            return True
        except Exception as e:
            print(f"(Could not install exifread automatically: {e})")
            print("(Run `pip install exifread` yourself if you want maker-note tags too.)")
            return False


# ============================================================
# SET THIS to the image you want to inspect
# ============================================================
IMAGE_PATH = Path(DATA_DIR, "Coralnet-Final", "Coral Garden", "2024_JUL_CG_S", "CG-S_T1", "GOPR3087.JPG")
# ============================================================


def print_section(title):
    print()
    print("=" * 70)
    print(title)
    print("=" * 70)


def show_pil_exif(image_path):
    from PIL import Image
    from PIL.ExifTags import TAGS, GPSTAGS

    print_section("PIL / Pillow EXIF tags")

    img = Image.open(image_path)
    print(f"Format: {img.format} | Mode: {img.mode} | Size: {img.size}")

    exif = img._getexif()
    if not exif:
        print("No EXIF data found via PIL.")
        return

    for tag_id, value in sorted(exif.items(), key=lambda kv: TAGS.get(kv[0], str(kv[0]))):
        tag = TAGS.get(tag_id, tag_id)

        if tag == "GPSInfo" and isinstance(value, dict):
            print(f"  {tag}:")
            for gps_id, gps_val in value.items():
                gps_tag = GPSTAGS.get(gps_id, gps_id)
                print(f"      {gps_tag}: {gps_val}")
        else:
            # truncate very long binary-looking values for readability
            display_val = value
            if isinstance(value, bytes):
                display_val = f"<{len(value)} bytes of binary data>"
            elif isinstance(value, str) and len(value) > 300:
                display_val = value[:300] + "... <truncated>"
            print(f"  {tag}: {display_val}")


def show_exifread_tags(image_path, have_exifread):
    if not have_exifread:
        return
    import exifread

    print_section("exifread tags (includes maker notes)")

    with open(image_path, "rb") as f:
        tags = exifread.process_file(f, details=True)

    if not tags:
        print("No tags found via exifread.")
        return

    for key in sorted(tags.keys()):
        value = tags[key]
        display_val = str(value)
        if len(display_val) > 300:
            display_val = display_val[:300] + "... <truncated>"
        print(f"  {key}: {display_val}")


def show_file_basics(image_path):
    import os

    print_section("File basics")
    size_bytes = os.path.getsize(image_path)
    print(f"  Path: {image_path}")
    print(f"  Size: {size_bytes:,} bytes ({size_bytes / (1024*1024):.2f} MB)")


def main():
    show_file_basics(IMAGE_PATH)
    have_exifread = ensure_exifread()
    show_pil_exif(IMAGE_PATH)
    show_exifread_tags(IMAGE_PATH, have_exifread)
    print()
    print("Done.")


if __name__ == "__main__":
    main()


File basics
  Path: C:\Users\Administrator\Documents\Thesis\data\Coralnet-Final\Coral Garden\2024_JUL_CG_S\CG-S_T1\GOPR3087.JPG
  Size: 5,993,839 bytes (5.72 MB)

PIL / Pillow EXIF tags
Format: JPEG | Mode: RGB | Size: (4924, 2936)
  ApertureValue: 2.5
  BodySerialNumber: C3441325549787               
  ColorSpace: 0
  ComponentsConfiguration: <4 bytes of binary data>
  CompressedBitsPerPixel: 0.004
  Contrast: 0
  CustomRendered: 0
  DateTime: 2024:07:02 13:15:11
  DateTimeDigitized: 2024:07:02 13:15:11
  DateTimeOriginal: 2024:07:02 13:15:11
  DeviceSettingDescription: <4 bytes of binary data>
  DigitalZoomRatio: 1.0
  ExifImageHeight: 2936
  ExifImageWidth: 4924
  ExifInteroperabilityOffset: 3313
  ExifOffset: 298
  ExifVersion: <4 bytes of binary data>
  ExposureBiasValue: 0.0
  ExposureIndex: 1.0666758371471003
  ExposureMode: 0
  ExposureProgram: 2
  ExposureTime: 0.0010395010395010396
  FNumber: 2.5
  FileSource: <1 bytes of binary data>
  Flash: 32
  FlashPixVersion: <4 bytes 

# **SECTION 2:** JULY ENTRIES CROPPING (OLD)

---

## **2.1** CROPPING OF ALL CORALNET DATASET

In [5]:
"""
crop_coralnet_quadrat.py

Applies a fixed-geometry crop to every image in the CoralNet-Dataset directory
tree, removing the environment outside the PVC quadrat rig from each frame
before the images are fed into the YOLOv8 / SAHI inference pipeline.

CROP GEOMETRY (all expressed as fractions of each image's own width/height,
recomputed per file so it still works if a handful of images have different
resolutions):

    LEFT_MARGIN_FRAC   = 0.025   # trim 2.5% off the left
    RIGHT_MARGIN_FRAC  = 0.025   # trim 2.5% off the right
    TOP_MARGIN_FRAC    = 0.025   # trim 2.5% off the top
    BOTTOM_CUTOFF_FRAC = 0.78    # keep only the top 78% of the frame

Where these numbers came from:
  - LEFT/RIGHT/TOP (2.5% each): matches the real coralnet.ucsd.edu platform's
    own "default annotation area" convention (X: 2.5-97.5% / Y: 2.5-97.5%),
    which is what the thesis's "central portion of each image" phrasing
    refers to. This is not an arbitrary choice; it mirrors CoralNet's own ROI
    default.
  - BOTTOM (78%): this is the "conservative universal cutoff" calibrated from
    4 sample images (GOPR3010/3337/3455/3514) spanning different divers/
    sites/transects. The PVC rig's top edge was observed between ~80% and
    ~93% down the frame across those 4 samples. 78% sits above the highest
    (most conservative) observed position with a small safety margin, so the
    rig, mount plate, and legs are excluded on every image. The trade-off:
    on images where the rig actually sits lower (e.g. ~90%+), real coral area
    between 78% and the rig's true position is discarded unnecessarily. If
    Stage 1 metrics later show this is too lossy, BOTTOM_CUTOFF_FRAC is the
    single number to revisit (see the "Larger manual calibration sample" or
    "Shape-based per-image detector" options discussed separately).

USAGE:
    1. Set INPUT_ROOT and OUTPUT_ROOT below.
    2. Run: python3 crop_coralnet_quadrat.py
    3. Check the printed summary for any per-file failures.

The full CoralNet-Dataset/ directory structure (Coral Garden, Giant Clam,
Haruka Piloto, Sapangan, each with their transect subfolders) is mirrored
exactly under OUTPUT_ROOT, so downstream scripts that expect the same
relative paths keep working unchanged.
"""

# ============================================================
# CONFIGURATION - adjust these if the crop needs recalibration
# ============================================================
INPUT_ROOT = Path(DATA_DIR, "Coralnet-raw")
OUTPUT_ROOT = Path(DATA_DIR, "Coralnet-Final")

LEFT_MARGIN_FRAC = 0.025
RIGHT_MARGIN_FRAC = 0.025
TOP_MARGIN_FRAC = 0.025
BOTTOM_CUTOFF_FRAC = 0.78

VALID_EXTENSIONS = {".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"}
JPEG_SAVE_QUALITY = 95

# Every crop percentage above was calibrated against GoPro HERO9 Black photos
# (5184x3888). At least one transect (Sapangan T1, ~46-48 images) was shot on
# a different camera at 4000x3000 - same 4:3 aspect ratio, but an unverified
# field of view. The percentages still apply geometrically to any resolution,
# but whether 78% is still the *correct* cutoff for a different camera's FOV
# is unconfirmed. Rather than silently assume it transfers, the script flags
# every non-standard-dimension file it processes so they can be spot-checked.
EXPECTED_DIMENSIONS = (5184, 3888)

# ============================================================


def compute_crop_box(width, height):
    """Returns (left, top, right, bottom) pixel coordinates for PIL's .crop()."""
    left = int(round(width * LEFT_MARGIN_FRAC))
    right = int(round(width * (1 - RIGHT_MARGIN_FRAC)))
    top = int(round(height * TOP_MARGIN_FRAC))
    bottom = int(round(height * BOTTOM_CUTOFF_FRAC))
    return left, top, right, bottom


def crop_image(input_path: Path, output_path: Path):
    """Crops a single image and saves it to output_path, carrying over the
    original's EXIF and DPI metadata (Pillow does NOT do this by default -
    a plain .save() silently strips EXIF, which loses camera model, focal
    length, timestamp, and GPS tags if present).
    Returns (success, error_message)."""
    try:
        with Image.open(input_path) as img:
            width, height = img.size
            left, top, right, bottom = compute_crop_box(width, height)

            if right <= left or bottom <= top:
                return False, f"invalid crop box for size {width}x{height}: ({left},{top},{right},{bottom})", None

            # Preserve original metadata before cropping (img.info is lost
            # once you call .crop(), so grab it from the source image now).
            exif_bytes = img.info.get("exif")
            dpi = img.info.get("dpi")

            cropped = img.crop((left, top, right, bottom))
            new_width, new_height = cropped.size
            output_path.parent.mkdir(parents=True, exist_ok=True)

            # Update the two EXIF fields that record pixel dimensions so they
            # don't go stale (they'd otherwise still say the ORIGINAL
            # 5184x3888 even though the actual cropped pixel data is smaller,
            # since a raw byte-copy of the EXIF block doesn't update them).
            if exif_bytes:
                try:
                    exif_dict = piexif.load(exif_bytes)
                    exif_dict["Exif"][piexif.ExifIFD.PixelXDimension] = new_width
                    exif_dict["Exif"][piexif.ExifIFD.PixelYDimension] = new_height
                    exif_bytes = piexif.dump(exif_dict)
                except Exception:
                    # If the EXIF block is malformed/unreadable by piexif,
                    # fall back to writing it through unchanged rather than
                    # dropping it or failing the whole file.
                    pass

            save_kwargs = {}
            if img.format == "JPEG" or output_path.suffix.lower() in (".jpg", ".jpeg"):
                save_kwargs["quality"] = JPEG_SAVE_QUALITY
            if exif_bytes:
                save_kwargs["exif"] = exif_bytes
            if dpi:
                save_kwargs["dpi"] = dpi

            cropped.save(output_path, **save_kwargs)
        return True, None, (width, height)
    except Exception as e:
        return False, str(e), None


def main():
    input_root = Path(INPUT_ROOT)
    output_root = Path(OUTPUT_ROOT)

    if not input_root.exists():
        print(f"ERROR: input root does not exist: {input_root}")
        sys.exit(1)

    all_files = sorted(
        p for p in input_root.rglob("*")
        if p.is_file() and p.suffix in VALID_EXTENSIONS
    )
    total = len(all_files)
    print(f"Found {total} images under {input_root}")
    print(f"Crop geometry: LEFT={LEFT_MARGIN_FRAC*100:.1f}% RIGHT={RIGHT_MARGIN_FRAC*100:.1f}% "
          f"TOP={TOP_MARGIN_FRAC*100:.1f}% BOTTOM_CUTOFF={BOTTOM_CUTOFF_FRAC*100:.1f}%")
    print()

    success_count = 0
    failed = []
    nonstandard_dims = []

    for i, img_path in enumerate(all_files, 1):
        rel_path = img_path.relative_to(input_root)
        output_path = output_root / rel_path
        ok, err, dims = crop_image(img_path, output_path)
        if ok:
            success_count += 1
            if dims and dims != EXPECTED_DIMENSIONS:
                nonstandard_dims.append((str(rel_path), dims))
        else:
            failed.append((str(rel_path), err))

        if i % 100 == 0 or i == total:
            print(f"  Processed {i}/{total}  ({success_count} succeeded, {len(failed)} failed)")

    print()
    print("=" * 60)
    print(f"DONE: {success_count}/{total} images cropped successfully")
    if failed:
        print(f"{len(failed)} FAILURE(S):")
        for rel_path, err in failed:
            print(f"  {rel_path}: {err}")

    if nonstandard_dims:
        print()
        print(f"WARNING: {len(nonstandard_dims)} file(s) were NOT {EXPECTED_DIMENSIONS[0]}x{EXPECTED_DIMENSIONS[1]}")
        print("(likely a different camera - crop percentages were only visually")
        print("verified against GoPro HERO9 samples, so these are UNVERIFIED.")
        print("Open a few of these specific cropped outputs and confirm the rig")
        print("is actually excluded before trusting this batch):")
        for rel_path, dims in nonstandard_dims[:20]:
            print(f"  {rel_path}: {dims[0]}x{dims[1]}")
        if len(nonstandard_dims) > 20:
            print(f"  ... and {len(nonstandard_dims) - 20} more")

    print()
    print(f"Output written to: {output_root}")


if __name__ == "__main__":
    main()

Found 900 images under C:\Users\Administrator\Documents\Thesis\data\Coralnet-raw
Crop geometry: LEFT=2.5% RIGHT=2.5% TOP=2.5% BOTTOM_CUTOFF=78.0%

  Processed 100/900  (100 succeeded, 0 failed)
  Processed 200/900  (200 succeeded, 0 failed)
  Processed 300/900  (300 succeeded, 0 failed)
  Processed 400/900  (400 succeeded, 0 failed)
  Processed 500/900  (500 succeeded, 0 failed)
  Processed 600/900  (600 succeeded, 0 failed)
  Processed 700/900  (700 succeeded, 0 failed)
  Processed 800/900  (800 succeeded, 0 failed)
  Processed 900/900  (900 succeeded, 0 failed)

DONE: 900/900 images cropped successfully

(likely a different camera - crop percentages were only visually
verified against GoPro HERO9 samples, so these are UNVERIFIED.
Open a few of these specific cropped outputs and confirm the rig
is actually excluded before trusting this batch):
  Sapangan (Deep Only)\2024_JUL_SAP_D\SAP-D_T1\P7031636.JPG: 4000x3000
  Sapangan (Deep Only)\2024_JUL_SAP_D\SAP-D_T1\P7031637.JPG: 4000x3000
 

---

## **2.2** CORALNET VALIDATION

In [10]:
"""
validate_coralnet_dataset.py

Cross-checks your LOCAL image folder (Coralnet-raw or Coralnet-Final) against
the official metadata.csv exported from CoralNet, to answer:

  1. How many local files actually match an official, curated CoralNet entry?
  2. Which official entries have NO local file (missing images)?
  3. Which local files have NO official entry (extra/uncurated frames)?
  4. Are there any images using a non-GoPro naming convention (different camera)?
  5. Do the 14 annotated images (from annotations.csv) actually exist locally?
  6. Do any local folder names fail to obviously correspond to a metadata
     station/transect tag (naming-convention drift)?

WHY MATCHING IS DONE BY FILENAME, NOT FOLDER PATH:
metadata.csv's "Name" column uses a flat, single-level folder tag (e.g.
"2024_JUL_CG_D_T2/GOPR3010.JPG") that does NOT match the deeper nested
physical folder structure on disk (e.g. "Coral Garden/JULY 2024_CG_D/
CG-D_T2/GOPR3010.JPG"). Trying to match on folder path would require
reconciling two genuinely different naming conventions. Instead, this script
matches on the base filename alone (e.g. "GOPR3010.JPG"), which is reliable
because every image was shot on a single GoPro body (confirmed via
BodySerialNumber in EXIF) EXCEPT for a small subset (see finding #4 below),
and no duplicate base filenames exist anywhere in metadata.csv.

USAGE:
    1. Set LOCAL_ROOT and METADATA_CSV (and optionally ANNOTATIONS_CSV) below.
    2. Run: python3 validate_coralnet_dataset.py
    3. Check the console summary, and the two CSVs written next to this
       script (missing_from_local.csv, extra_in_local.csv) for full detail.
"""

import re
import sys
from pathlib import Path
from collections import Counter

import pandas as pd

# ============================================================
# CONFIGURATION - adjust these paths
# ============================================================
LOCAL_ROOT = Path(DATA_DIR, "Coralnet-raw")          # or Coralnet-Final
METADATA_CSV = Path(DATA_DIR, "Coralnet-raw", "metadata.csv")
ANNOTATIONS_CSV = Path(DATA_DIR, "Coralnet-raw", "annotations.csv")  # optional; set to None to skip

VALID_EXTENSIONS = {".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"}

OUTPUT_DIR = "."  # where missing_from_local.csv / extra_in_local.csv are written

# ============================================================


def extract_basefile(name_field: str) -> str:
    """
    Pulls just the filename (last path segment) out of metadata.csv's
    'Name' column, handling the few rows that use '_' instead of '/' as
    a separator (e.g. '2024_JUL_HPFS_S_T2_GOPR3739.JPG').
    """
    # Standard case: folder/FILENAME.ext
    if "/" in name_field:
        return name_field.split("/")[-1].strip().upper()
    # Fallback: pull the trailing GOPR#### / P###### / similar token + extension
    m = re.search(r"([A-Za-z0-9]+\.\w+)$", name_field)
    return (m.group(1) if m else name_field).strip().upper()


def classify_camera_pattern(basefile: str) -> str:
    """Buckets a filename by naming convention, to flag non-GoPro cameras."""
    stem = basefile.upper()
    if stem.startswith("GOPR") or stem.startswith("GP"):
        return "GOPRO_STANDARD"
    if re.match(r"^P\d+\.", stem):
        return "OTHER_CAMERA_P-PREFIX"
    return "OTHER_UNRECOGNIZED"


def load_metadata(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["basefile"] = df["Name"].apply(extract_basefile)
    df["camera_pattern"] = df["basefile"].apply(classify_camera_pattern)
    df["month"] = df["Name"].apply(lambda x: "JUL" if "JUL" in x.upper() else ("AUG" if "AUG" in x.upper() else "UNKNOWN"))
    return df


def walk_local_files(root: Path):
    """Returns list of (relative_path, basefile) for every image file under root."""
    found = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in VALID_EXTENSIONS:
            found.append((str(p.relative_to(root)), p.name.strip().upper()))
    return found


def main():
    local_root = Path(LOCAL_ROOT)
    metadata_csv = Path(METADATA_CSV)

    if not local_root.exists():
        print(f"ERROR: LOCAL_ROOT does not exist: {local_root}")
        sys.exit(1)
    if not metadata_csv.exists():
        print(f"ERROR: METADATA_CSV does not exist: {metadata_csv}")
        sys.exit(1)

    print("=" * 70)
    print("STEP 1: Loading metadata.csv")
    print("=" * 70)
    meta = load_metadata(metadata_csv)
    print(f"Official entries in metadata.csv: {len(meta)}")
    print(meta["month"].value_counts().to_string())
    print()
    print("Camera naming pattern breakdown (official):")
    print(meta["camera_pattern"].value_counts().to_string())
    non_gopro = meta[meta["camera_pattern"] != "GOPRO_STANDARD"]
    if len(non_gopro) > 0:
        print()
        print(f"  --> {len(non_gopro)} officially-listed images do NOT use standard GoPro")
        print("      naming. These likely came from a different camera and may need")
        print("      separate handling in the crop script (different resolution/FOV).")
        print("      Affected stations/transects:")
        for (station, transect), grp in non_gopro.groupby(["Monitoring Station", "Transect number"]):
            print(f"        {station} {transect}: {len(grp)} image(s), e.g. {grp['basefile'].iloc[0]}")
    print()

    print("=" * 70)
    print("STEP 2: Walking local folder")
    print("=" * 70)
    local_files = walk_local_files(local_root)
    print(f"Local image files found: {len(local_files)}")
    local_camera_patterns = Counter(classify_camera_pattern(bf) for _, bf in local_files)
    print("Camera naming pattern breakdown (local):")
    for k, v in local_camera_patterns.items():
        print(f"  {k}: {v}")
    print()

    # check for local duplicate basefiles (same filename in two different folders)
    local_basefile_to_paths = {}
    for rel_path, basefile in local_files:
        local_basefile_to_paths.setdefault(basefile, []).append(rel_path)
    local_dupes = {k: v for k, v in local_basefile_to_paths.items() if len(v) > 1}
    if local_dupes:
        print(f"WARNING: {len(local_dupes)} filename(s) appear in MORE THAN ONE local folder:")
        for k, v in list(local_dupes.items())[:10]:
            print(f"  {k}: {v}")
        print()

    print("=" * 70)
    print("STEP 3: Cross-matching local files against metadata.csv")
    print("=" * 70)
    official_basefiles = set(meta["basefile"])
    local_basefiles = set(local_basefile_to_paths.keys())

    matched = official_basefiles & local_basefiles
    missing_from_local = official_basefiles - local_basefiles
    extra_in_local = local_basefiles - official_basefiles

    print(f"Matched (official AND local):     {len(matched)}")
    print(f"Missing (official, no local file): {len(missing_from_local)}")
    print(f"Extra (local, not in metadata):    {len(extra_in_local)}")
    print()

    missing_df = meta[meta["basefile"].isin(missing_from_local)]
    print("Missing-from-local breakdown by month:")
    print(missing_df["month"].value_counts().to_string())
    print()
    print("Missing-from-local breakdown by station (top 10):")
    print(missing_df["Monitoring Station"].value_counts().head(10).to_string())
    print()

    if extra_in_local:
        print(f"Extra-in-local examples (up to 10 of {len(extra_in_local)}):")
        for bf in list(extra_in_local)[:10]:
            print(f"  {bf}  ->  {local_basefile_to_paths[bf][0]}")
        print()

    # write full detail to CSV
    out_dir = Path(OUTPUT_DIR)
    missing_out = out_dir / "missing_from_local.csv"
    extra_out = out_dir / "extra_in_local.csv"

    missing_df.to_csv(missing_out, index=False)

    extra_rows = [
        {"basefile": bf, "local_path": local_basefile_to_paths[bf][0]}
        for bf in sorted(extra_in_local)
    ]
    pd.DataFrame(extra_rows).to_csv(extra_out, index=False)
    print(f"Full missing list written to: {missing_out}")
    print(f"Full extra list written to:   {extra_out}")
    print()

    # ------------------------------------------------------------------
    # STEP 4: cross-check annotations.csv, if provided
    # ------------------------------------------------------------------
    if ANNOTATIONS_CSV:
        ann_path = Path(ANNOTATIONS_CSV)
        if ann_path.exists():
            print("=" * 70)
            print("STEP 4: Checking annotated images against local files")
            print("=" * 70)
            ann = pd.read_csv(ann_path)
            ann["basefile"] = ann["Name"].apply(extract_basefile)
            annotated_basefiles = set(ann["basefile"])
            print(f"Unique annotated images in annotations.csv: {len(annotated_basefiles)}")
            present = annotated_basefiles & local_basefiles
            absent = annotated_basefiles - local_basefiles
            print(f"  Present locally: {len(present)}")
            print(f"  Absent locally:  {len(absent)}")
            if absent:
                print(f"  Missing annotated files: {sorted(absent)}")
            print()
        else:
            print(f"(ANNOTATIONS_CSV set but not found at {ann_path}, skipping step 4)")
            print()

    # ------------------------------------------------------------------
    # STEP 5: folder-naming-convention sanity check
    # ------------------------------------------------------------------
    print("=" * 70)
    print("STEP 5: Local folder name vs. metadata station/transect tags")
    print("=" * 70)
    local_leaf_folders = sorted(set(Path(rel).parent.name for rel, _ in local_files))
    metadata_folder_tags = sorted(set(meta["Name"].apply(lambda x: x.split("/")[0] if "/" in x else "UNKNOWN")))
    print("Local leaf folder names found:")
    for f in local_leaf_folders:
        print(f"  {f}")
    print()
    print("metadata.csv folder tags (sample of up to 15):")
    for f in metadata_folder_tags[:15]:
        print(f"  {f}")
    print()
    print("NOTE: these two naming conventions are expected to look different")
    print("(this is exactly why matching is done by filename, not folder path).")
    print("This section is for your own visual sanity check of naming drift,")
    print("e.g. confirming 'Haruka Piloto' vs metadata's 'Harka Piloto' spelling.")

    print()
    print("=" * 70)
    print("DONE")
    print("=" * 70)


if __name__ == "__main__":
    main()

STEP 1: Loading metadata.csv
Official entries in metadata.csv: 1798
month
JUL    900
AUG    898

Camera naming pattern breakdown (official):
camera_pattern
GOPRO_STANDARD           1752
OTHER_CAMERA_P-PREFIX      46

  --> 46 officially-listed images do NOT use standard GoPro
      naming. These likely came from a different camera and may need
      separate handling in the crop script (different resolution/FOV).
      Affected stations/transects:
        Sapangan T1: 46 image(s), e.g. P7031636.JPG

STEP 2: Walking local folder
Local image files found: 985
Camera naming pattern breakdown (local):
  GOPRO_STANDARD: 934
  OTHER_UNRECOGNIZED: 3
  OTHER_CAMERA_P-PREFIX: 48

STEP 3: Cross-matching local files against metadata.csv
Matched (official AND local):     900
Missing (official, no local file): 898
Extra (local, not in metadata):    85

Missing-from-local breakdown by month:
month
AUG    898

Missing-from-local breakdown by station (top 10):
Monitoring Station
Coral Garden S    150
G

---

## **2.3** REMOVE NON-CURATED EXTRAS FROM CORALNET

In [4]:
"""
remove_noncurated_extras.py

Moves (never deletes - this is reversible) the images identified in
extra_in_local.csv as NOT part of the official 900-image July CoralNet
source out of your raw folder, based on your confirmation that these are
diver transect-marker/setup shots with no reason to be in metadata.csv.

WHY MOVE INSTEAD OF DELETE:
If it later turns out one of these was needed after all, you can just move
the folder back. Deleting is one-way; moving costs nothing extra and is
just as clean once you're done.

ONE IMPORTANT EXCEPTION - files this script will NOT touch automatically:
extra_in_local.csv also lists a few files with UUID-style names
(e.g. "91D16B35-B90C-49B0-924C-493F70BFE5E8.JPG") instead of the camera-native
GOPR#### / P####### pattern. Those don't match the "diver transect-marker
photo" explanation - that naming convention isn't something a GoPro or the
Sapangan alternate camera would ever produce, so they're probably something
else entirely (a phone photo, a screenshot, a synced/renamed duplicate).
This script lists them separately at the end and leaves them exactly where
they are, so you can look at each one yourself before deciding what to do
with it.

USAGE:
    1. Set RAW_ROOT and EXTRA_CSV below (EXTRA_CSV is the file
       validate_coralnet_dataset.py already wrote for you).
    2. Run: python3 remove_noncurated_extras.py
    3. Check the printed summary. Moved files land in EXCLUDED_ROOT,
       preserving their original relative folder structure.
"""

# ============================================================
# CONFIGURATION
# ============================================================
RAW_ROOT = Path(DATA_DIR, "Coralnet-raw")
EXTRA_CSV = Path(PROJECT_ROOT, "extra_in_local.csv")
EXCLUDED_ROOT = Path(DATA_DIR, "Coralnet-excluded")  #"/path/to/Coralnet-raw-excluded" (sibling folder, NOT inside RAW_ROOT)


# ============================================================


def is_camera_native_name(basefile: str) -> bool:
    """True for GOPR####.ext or P#######.ext style names; False for anything
    else (e.g. UUID-style names), which get flagged instead of moved."""
    return bool(re.match(r"^(GOPR|GP|P\d+)", basefile.upper()))


def main():
    raw_root = Path(RAW_ROOT)
    excluded_root = Path(EXCLUDED_ROOT)
    extra_csv = Path(EXTRA_CSV)

    if not raw_root.exists():
        print(f"ERROR: RAW_ROOT does not exist: {raw_root}")
        sys.exit(1)
    if not extra_csv.exists():
        print(f"ERROR: EXTRA_CSV does not exist: {extra_csv}")
        print("(Run validate_coralnet_dataset.py first - it writes this file.)")
        sys.exit(1)

    df = pd.read_csv(extra_csv)
    print(f"Loaded {len(df)} entries from {extra_csv.name}")
    print()

    to_move = []
    to_flag = []
    for _, row in df.iterrows():
        basefile = str(row["basefile"])
        local_path = str(row["local_path"])
        if is_camera_native_name(basefile):
            to_move.append(local_path)
        else:
            to_flag.append(local_path)

    print(f"Camera-native extras to move (transect-marker shots): {len(to_move)}")
    print(f"Non-standard-name files flagged for manual review:     {len(to_flag)}")
    print()

    moved = 0
    move_errors = []
    for rel_path in to_move:
        # local_path in the CSV uses the OS separator from when it was
        # written; normalize both slash styles just in case.
        rel_path_norm = rel_path.replace("\\", "/")
        src = raw_root / rel_path_norm
        dst = excluded_root / rel_path_norm
        if not src.exists():
            move_errors.append((rel_path, "source file not found (already moved?)"))
            continue
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            moved += 1
        except Exception as e:
            move_errors.append((rel_path, str(e)))

    print("=" * 70)
    print(f"Moved {moved}/{len(to_move)} camera-native extra files to:")
    print(f"  {excluded_root}")
    if move_errors:
        print()
        print(f"{len(move_errors)} file(s) could not be moved:")
        for rel_path, err in move_errors:
            print(f"  {rel_path}: {err}")

    if to_flag:
        print()
        print("=" * 70)
        print("NOT moved - please look at these yourself, they don't match the")
        print("'diver transect-marker photo' explanation:")
        for rel_path in to_flag:
            print(f"  {rel_path}")

    print()
    print("=" * 70)
    print("DONE")
    print("=" * 70)


if __name__ == "__main__":
    main()

Loaded 85 entries from extra_in_local.csv

Camera-native extras to move (transect-marker shots): 82
Non-standard-name files flagged for manual review:     3

Moved 82/82 camera-native extra files to:
  C:\Users\Administrator\Documents\Thesis\data\Coralnet-excluded

NOT moved - please look at these yourself, they don't match the
'diver transect-marker photo' explanation:
  Haruka Piloto\2024_HP_JUL_S\HP_S_T2\91d16b35-b90c-49b0-924c-493f70bfe5e8.jpg
  Haruka Piloto\2024_HP_JUL_S\HP_S_T1\bb647eb0-a7fa-44f9-a8b0-35cee2bc1435.jpg
  Haruka Piloto\2024_HP_JUL_S\HP_S_T1\e2db8ae8-2bd2-4edf-85fc-6dd4ffe8beb0.jpg

DONE


---

# **SECTION 3:** AUGUST ENTRIES CROPPING

## rename_harka_piloto.py

In [3]:
"""
rename_harka_piloto.py

Corrects the local "Haruka Piloto" folder name to match metadata.csv's
actual station name ("Harka Piloto D"/"Harka Piloto S") -- confirmed by
reading metadata.csv directly, not assumed.

Scope: Coralnet-raw, Coralnet-Final, and Coralnet-excluded only. Does NOT
touch Coralnet-Patches or Refined-Patches -- those still reference
"Haruka Piloto" in patches_manifest.csv / patches_manifest_refined.csv AND
on disk, and Refined-Patches holds manually-curated output (1,052 patches
already hand-deleted). Renaming that safely means rewriting manifest rows,
not just moving a folder -- treated as a separate follow-up, not done here.

Idempotent: safe to re-run. Does nothing for any root where "Haruka Piloto"
no longer exists (already renamed, or never existed there).
"""

RENAME_TARGETS = [
    Path(DATA_DIR, "Coralnet-raw"),
    Path(DATA_DIR, "Coralnet-Final"),
    Path(DATA_DIR, "Coralnet-excluded"),  # cosmetic consistency only -- not read downstream
]

def rename_haruka_to_harka(root: Path):
    old = root / "Haruka Piloto"
    new = root / "Harka Piloto"
    if not old.exists():
        print(f"[skip] {old} not found (already renamed, or nothing here yet)")
        return
    if new.exists():
        raise RuntimeError(
            f"Both {old} and {new} exist under {root} -- refusing to auto-merge. "
            f"Resolve by hand before re-running."
        )
    old.rename(new)
    print(f"[renamed] {old}  ->  {new}")

for root in RENAME_TARGETS:
    rename_haruka_to_harka(root)

[skip] C:\Users\Admin\Documents\Thesis\data\Coralnet-raw\Haruka Piloto not found (already renamed, or nothing here yet)
[skip] C:\Users\Admin\Documents\Thesis\data\Coralnet-Final\Haruka Piloto not found (already renamed, or nothing here yet)
[skip] C:\Users\Admin\Documents\Thesis\data\Coralnet-excluded\Haruka Piloto not found (already renamed, or nothing here yet)


## validate_coralnet_dataset_v2.py

In [4]:
"""
validate_coralnet_dataset_v2.py

Same matching logic as the original validate_coralnet_dataset.py (filename-
based, not folder-path-based -- metadata.csv's folder tags and the real
directory tree are still two different conventions). Re-run now that
Coralnet-raw physically contains both July and August files, post-rename.

Expected if the acquisition + rename went cleanly: matched climbs from
900 -> ~1798, missing drops from 898 -> ~0. extra_in_local should only grow
if the August batch itself contains uncurated extras the same way July did.

NOTE: the 3 UUID-named files flagged (not moved) in the original run are
still sitting in Coralnet-raw and will reappear here as OTHER_UNRECOGNIZED
extras -- expected, not a new issue, still needs your manual look.

Writes to NEW filenames so the original July-only extra_in_local.csv /
missing_from_local.csv stay on record as an audit trail.
"""

import re
from collections import Counter
import pandas as pd

LOCAL_ROOT = Path(DATA_DIR, "Coralnet-raw")
METADATA_CSV = Path(DATA_DIR, "Coralnet-raw", "metadata.csv")
ANNOTATIONS_CSV = Path(DATA_DIR, "Coralnet-raw", "annotations.csv")

VALID_EXTENSIONS = {".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"}
OUTPUT_DIR = "."
MISSING_OUT_NAME = "missing_from_local_full.csv"
EXTRA_OUT_NAME = "extra_in_local_full.csv"


def extract_basefile(name_field: str) -> str:
    if "/" in name_field:
        return name_field.split("/")[-1].strip().upper()
    m = re.search(r"([A-Za-z0-9]+\.\w+)$", name_field)
    return (m.group(1) if m else name_field).strip().upper()


def classify_camera_pattern(basefile: str) -> str:
    stem = basefile.upper()
    if stem.startswith("GOPR") or stem.startswith("GP"):
        return "GOPRO_STANDARD"
    if re.match(r"^P\d+\.", stem):
        return "OTHER_CAMERA_P-PREFIX"
    return "OTHER_UNRECOGNIZED"


def load_metadata(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    df["basefile"] = df["Name"].apply(extract_basefile)
    df["camera_pattern"] = df["basefile"].apply(classify_camera_pattern)
    df["month"] = df["Name"].apply(lambda x: "JUL" if "JUL" in x.upper() else ("AUG" if "AUG" in x.upper() else "UNKNOWN"))
    return df


def walk_local_files(root: Path):
    found = []
    for p in root.rglob("*"):
        if p.is_file() and p.suffix in VALID_EXTENSIONS:
            found.append((str(p.relative_to(root)), p.name.strip().upper()))
    return found


def main():
    local_root = Path(LOCAL_ROOT)
    metadata_csv = Path(METADATA_CSV)

    if not local_root.exists():
        print(f"ERROR: LOCAL_ROOT does not exist: {local_root}")
        return
    if not metadata_csv.exists():
        print(f"ERROR: METADATA_CSV does not exist: {metadata_csv}")
        return

    print("=" * 70)
    print("STEP 1: Loading metadata.csv")
    print("=" * 70)
    meta = load_metadata(metadata_csv)
    print(f"Official entries in metadata.csv: {len(meta)}")
    print(meta["month"].value_counts().to_string())
    print()
    print("Camera naming pattern breakdown (official):")
    print(meta["camera_pattern"].value_counts().to_string())
    non_gopro = meta[meta["camera_pattern"] != "GOPRO_STANDARD"]
    if len(non_gopro) > 0:
        print()
        print(f"  --> {len(non_gopro)} officially-listed images do NOT use standard GoPro naming.")
        print("      Affected stations/transects:")
        for (station, transect), grp in non_gopro.groupby(["Monitoring Station", "Transect number"]):
            print(f"        {station} {transect}: {len(grp)} image(s), e.g. {grp['basefile'].iloc[0]}")
    print()

    print("=" * 70)
    print("STEP 2: Walking local folder (post-August-merge, post-Harka-rename)")
    print("=" * 70)
    local_files = walk_local_files(local_root)
    print(f"Local image files found: {len(local_files)}")
    local_camera_patterns = Counter(classify_camera_pattern(bf) for _, bf in local_files)
    print("Camera naming pattern breakdown (local):")
    for k, v in local_camera_patterns.items():
        print(f"  {k}: {v}")
    print()

    local_basefile_to_paths = {}
    for rel_path, basefile in local_files:
        local_basefile_to_paths.setdefault(basefile, []).append(rel_path)
    local_dupes = {k: v for k, v in local_basefile_to_paths.items() if len(v) > 1}
    if local_dupes:
        print(f"WARNING: {len(local_dupes)} filename(s) appear in MORE THAN ONE local folder:")
        for k, v in list(local_dupes.items())[:10]:
            print(f"  {k}: {v}")
        print()

    print("=" * 70)
    print("STEP 3: Cross-matching local files against metadata.csv")
    print("=" * 70)
    official_basefiles = set(meta["basefile"])
    local_basefiles = set(local_basefile_to_paths.keys())

    matched = official_basefiles & local_basefiles
    missing_from_local = official_basefiles - local_basefiles
    extra_in_local = local_basefiles - official_basefiles

    print(f"Matched (official AND local):     {len(matched)}")
    print(f"Missing (official, no local file): {len(missing_from_local)}")
    print(f"Extra (local, not in metadata):    {len(extra_in_local)}")
    print()

    missing_df = meta[meta["basefile"].isin(missing_from_local)]
    if len(missing_df) > 0:
        print("Missing-from-local breakdown by month:")
        print(missing_df["month"].value_counts().to_string())
        print()
        print("Missing-from-local breakdown by station (top 10):")
        print(missing_df["Monitoring Station"].value_counts().head(10).to_string())
        print()

    if extra_in_local:
        print(f"Extra-in-local examples (up to 10 of {len(extra_in_local)}):")
        for bf in list(extra_in_local)[:10]:
            print(f"  {bf}  ->  {local_basefile_to_paths[bf][0]}")
        print()

    out_dir = Path(OUTPUT_DIR)
    missing_out = out_dir / MISSING_OUT_NAME
    extra_out = out_dir / EXTRA_OUT_NAME

    missing_df.to_csv(missing_out, index=False)
    extra_rows = [
        {"basefile": bf, "local_path": local_basefile_to_paths[bf][0]}
        for bf in sorted(extra_in_local)
    ]
    pd.DataFrame(extra_rows).to_csv(extra_out, index=False)
    print(f"Full missing list written to: {missing_out}")
    print(f"Full extra list written to:   {extra_out}")
    print()

    if ANNOTATIONS_CSV and Path(ANNOTATIONS_CSV).exists():
        print("=" * 70)
        print("STEP 4: Checking annotated images against local files")
        print("=" * 70)
        ann = pd.read_csv(ANNOTATIONS_CSV)
        ann["basefile"] = ann["Name"].apply(extract_basefile)
        annotated_basefiles = set(ann["basefile"])
        present = annotated_basefiles & local_basefiles
        absent = annotated_basefiles - local_basefiles
        print(f"Unique annotated images in annotations.csv: {len(annotated_basefiles)}")
        print(f"  Present locally: {len(present)}")
        print(f"  Absent locally:  {len(absent)}")
        print()

    print("=" * 70)
    print("DONE")
    print("=" * 70)


if __name__ == "__main__":
    main()

STEP 1: Loading metadata.csv
Official entries in metadata.csv: 1798
month
JUL    900
AUG    898

Camera naming pattern breakdown (official):
camera_pattern
GOPRO_STANDARD           1752
OTHER_CAMERA_P-PREFIX      46

  --> 46 officially-listed images do NOT use standard GoPro naming.
      Affected stations/transects:
        Sapangan T1: 46 image(s), e.g. P7031636.JPG

STEP 2: Walking local folder (post-August-merge, post-Harka-rename)
Local image files found: 1903
Camera naming pattern breakdown (local):
  GOPRO_STANDARD: 1857
  OTHER_CAMERA_P-PREFIX: 46

STEP 3: Cross-matching local files against metadata.csv
Matched (official AND local):     1797
Missing (official, no local file): 1
Extra (local, not in metadata):    106

Missing-from-local breakdown by month:
month
AUG    1

Missing-from-local breakdown by station (top 10):
Monitoring Station
Coral Garden S    1

Extra-in-local examples (up to 10 of 106):
  GOPR5414.JPG  ->  Harka Piloto\2024_AUG_HPFS_D\2024_AUG_HPFS_D_T3\extra\GO

## remove_noncurated_extras.py

In [5]:
"""
remove_noncurated_extras.py -- unchanged logic, updated EXTRA_CSV only.

Reads extra_in_local_full.csv (from Cell 2, post-rename) instead of the
original July-only extra_in_local.csv. Same behavior as before: moves
(never deletes) camera-native-named extras to Coralnet-excluded; leaves
UUID-style names in place, flagged for manual review.

The 3 already-known UUID files WILL reappear in the "flagged, not moved"
list below -- that's expected, not a regression; they were never resolved.
"""

RAW_ROOT = Path(DATA_DIR, "Coralnet-raw")
EXTRA_CSV = Path(PROJECT_ROOT, "extra_in_local_full.csv")
EXCLUDED_ROOT = Path(DATA_DIR, "Coralnet-excluded")


def is_camera_native_name(basefile: str) -> bool:
    return bool(re.match(r"^(GOPR|GP|P\d+)", basefile.upper()))


def main():
    raw_root = Path(RAW_ROOT)
    excluded_root = Path(EXCLUDED_ROOT)
    extra_csv = Path(EXTRA_CSV)

    if not raw_root.exists():
        print(f"ERROR: RAW_ROOT does not exist: {raw_root}")
        return
    if not extra_csv.exists():
        print(f"ERROR: EXTRA_CSV does not exist: {extra_csv}")
        print("(Run validate_coralnet_dataset_v2.py first -- it writes this file.)")
        return

    df = pd.read_csv(extra_csv)
    print(f"Loaded {len(df)} entries from {extra_csv.name}")
    print()

    to_move, to_flag = [], []
    for _, row in df.iterrows():
        basefile = str(row["basefile"])
        local_path = str(row["local_path"])
        (to_move if is_camera_native_name(basefile) else to_flag).append(local_path)

    print(f"Camera-native extras to move (transect-marker shots): {len(to_move)}")
    print(f"Non-standard-name files flagged for manual review:     {len(to_flag)}")
    print()

    moved = 0
    move_errors = []
    for rel_path in to_move:
        rel_path_norm = rel_path.replace("\\", "/")
        src = raw_root / rel_path_norm
        dst = excluded_root / rel_path_norm
        if not src.exists():
            move_errors.append((rel_path, "source file not found (already moved?)"))
            continue
        try:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(src), str(dst))
            moved += 1
        except Exception as e:
            move_errors.append((rel_path, str(e)))

    print("=" * 70)
    print(f"Moved {moved}/{len(to_move)} camera-native extra files to:")
    print(f"  {excluded_root}")
    if move_errors:
        print()
        print(f"{len(move_errors)} file(s) could not be moved:")
        for rel_path, err in move_errors:
            print(f"  {rel_path}: {err}")

    if to_flag:
        print()
        print("=" * 70)
        print("NOT moved -- please look at these yourself:")
        for rel_path in to_flag:
            print(f"  {rel_path}")

    print()
    print("=" * 70)
    print("DONE")
    print("=" * 70)


if __name__ == "__main__":
    main()

Loaded 106 entries from extra_in_local_full.csv

Camera-native extras to move (transect-marker shots): 106
Non-standard-name files flagged for manual review:     0

Moved 106/106 camera-native extra files to:
  C:\Users\Admin\Documents\Thesis\data\Coralnet-excluded

DONE


## crop_coralnet_quadrat_v2.py

In [6]:
"""
crop_coralnet_quadrat_v2.py

Same crop geometry and EXIF-preservation as the original script -- LEFT/RIGHT/
TOP 2.5%, BOTTOM 78%, not reopened here. Difference: walks the FULL,
post-rename, post-extras-removal Coralnet-raw tree (~1,713 files -- 1,798
official minus whatever the August round's own extras turn out to be), and
SKIPS any file whose output path already exists under Coralnet-Final. That
skip is what makes "add the August entries to the existing Coralnet-Final"
actually true, rather than a full, pointless 1,798-image re-crop.

Run this AFTER Cells 1-3, so already-cropped July files are found under
their corrected "Harka Piloto" path (and skipped), and any August-round
extras never get cropped in the first place.
"""

INPUT_ROOT = Path(DATA_DIR, "Coralnet-raw")
OUTPUT_ROOT = Path(DATA_DIR, "Coralnet-Final")

LEFT_MARGIN_FRAC = 0.025
RIGHT_MARGIN_FRAC = 0.025
TOP_MARGIN_FRAC = 0.025
BOTTOM_CUTOFF_FRAC = 0.78

VALID_EXTENSIONS = {".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG"}
JPEG_SAVE_QUALITY = 95
EXPECTED_DIMENSIONS = (5184, 3888)


def compute_crop_box(width, height):
    left = int(round(width * LEFT_MARGIN_FRAC))
    right = int(round(width * (1 - RIGHT_MARGIN_FRAC)))
    top = int(round(height * TOP_MARGIN_FRAC))
    bottom = int(round(height * BOTTOM_CUTOFF_FRAC))
    return left, top, right, bottom


def crop_image(input_path: Path, output_path: Path):
    try:
        with Image.open(input_path) as img:
            width, height = img.size
            left, top, right, bottom = compute_crop_box(width, height)
            if right <= left or bottom <= top:
                return False, f"invalid crop box for size {width}x{height}", None

            exif_bytes = img.info.get("exif")
            dpi = img.info.get("dpi")

            cropped = img.crop((left, top, right, bottom))
            new_width, new_height = cropped.size
            output_path.parent.mkdir(parents=True, exist_ok=True)

            if exif_bytes:
                try:
                    exif_dict = piexif.load(exif_bytes)
                    exif_dict["Exif"][piexif.ExifIFD.PixelXDimension] = new_width
                    exif_dict["Exif"][piexif.ExifIFD.PixelYDimension] = new_height
                    exif_bytes = piexif.dump(exif_dict)
                except Exception:
                    pass

            save_kwargs = {}
            if img.format == "JPEG" or output_path.suffix.lower() in (".jpg", ".jpeg"):
                save_kwargs["quality"] = JPEG_SAVE_QUALITY
            if exif_bytes:
                save_kwargs["exif"] = exif_bytes
            if dpi:
                save_kwargs["dpi"] = dpi

            cropped.save(output_path, **save_kwargs)
        return True, None, (width, height)
    except Exception as e:
        return False, str(e), None


def main():
    input_root = Path(INPUT_ROOT)
    output_root = Path(OUTPUT_ROOT)

    if not input_root.exists():
        print(f"ERROR: input root does not exist: {input_root}")
        return

    all_files = sorted(
        p for p in input_root.rglob("*")
        if p.is_file() and p.suffix in VALID_EXTENSIONS
    )
    total_found = len(all_files)

    remaining = []
    already_done = 0
    for p in all_files:
        rel_path = p.relative_to(input_root)
        if (output_root / rel_path).exists():
            already_done += 1
        else:
            remaining.append(p)

    print(f"Found {total_found} images under {input_root}")
    print(f"Already cropped (present under {output_root}): {already_done}")
    print(f"Remaining to crop: {len(remaining)}")
    print(f"Crop geometry: LEFT={LEFT_MARGIN_FRAC*100:.1f}% RIGHT={RIGHT_MARGIN_FRAC*100:.1f}% "
          f"TOP={TOP_MARGIN_FRAC*100:.1f}% BOTTOM_CUTOFF={BOTTOM_CUTOFF_FRAC*100:.1f}%")
    print()

    success_count = 0
    failed = []
    nonstandard_dims = []

    for i, img_path in enumerate(remaining, 1):
        rel_path = img_path.relative_to(input_root)
        output_path = output_root / rel_path
        ok, err, dims = crop_image(img_path, output_path)
        if ok:
            success_count += 1
            if dims and dims != EXPECTED_DIMENSIONS:
                nonstandard_dims.append((str(rel_path), dims))
        else:
            failed.append((str(rel_path), err))

        if i % 100 == 0 or i == len(remaining):
            print(f"  Processed {i}/{len(remaining)}  ({success_count} succeeded, {len(failed)} failed)")

    print()
    print("=" * 60)
    print(f"DONE: {success_count}/{len(remaining)} new images cropped successfully")
    print(f"({already_done} were already present and skipped -- not re-cropped)")
    if failed:
        print(f"{len(failed)} FAILURE(S):")
        for rel_path, err in failed:
            print(f"  {rel_path}: {err}")

    if nonstandard_dims:
        print()
        print(f"WARNING: {len(nonstandard_dims)} newly-cropped file(s) were NOT {EXPECTED_DIMENSIONS[0]}x{EXPECTED_DIMENSIONS[1]}")
        print("(open a few of these specific outputs and confirm the rig is")
        print("actually excluded before trusting this batch -- same caveat as")
        print("applied to Sapangan T1 originally):")
        for rel_path, dims in nonstandard_dims[:20]:
            print(f"  {rel_path}: {dims[0]}x{dims[1]}")
        if len(nonstandard_dims) > 20:
            print(f"  ... and {len(nonstandard_dims) - 20} more")

    print()
    print(f"Output written to: {output_root}")


if __name__ == "__main__":
    main()

Found 1797 images under C:\Users\Admin\Documents\Thesis\data\Coralnet-raw
Already cropped (present under C:\Users\Admin\Documents\Thesis\data\Coralnet-Final): 900
Remaining to crop: 897
Crop geometry: LEFT=2.5% RIGHT=2.5% TOP=2.5% BOTTOM_CUTOFF=78.0%

  Processed 100/897  (100 succeeded, 0 failed)
  Processed 200/897  (200 succeeded, 0 failed)
  Processed 300/897  (300 succeeded, 0 failed)
  Processed 400/897  (400 succeeded, 0 failed)
  Processed 500/897  (500 succeeded, 0 failed)
  Processed 600/897  (600 succeeded, 0 failed)
  Processed 700/897  (700 succeeded, 0 failed)
  Processed 800/897  (800 succeeded, 0 failed)
  Processed 897/897  (897 succeeded, 0 failed)

DONE: 897/897 new images cropped successfully
(900 were already present and skipped -- not re-cropped)

Output written to: C:\Users\Admin\Documents\Thesis\data\Coralnet-Final
